# 08 · Chunking 高级

> 基础切分只保证“长度合适”，不保证“一块讲一件事”。高级切分让 chunk 语义内聚，显著提升检索命中率。

**本文件覆盖知识点**：Semantic Chunking / Sentence Window / Parent-Child / Hierarchical / Recursive / Contextual Chunking / Proposition / Document-Structure / Markdown / Code / Table Chunking

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 为什么需要语义化切分

```text
差:  chunk1=[....Redis采用单线程模]
     chunk2=[型...]          ← 一个概念被切成两半，谁都不完整
好:  chunk=[Redis 采用单线程事件循环模型，负责处理命令请求]  ← 一个语义单元
```

目标：**一个 chunk ≈ 一个相对独立的语义单元**（一条事实、一个论点、一段说明）。

## 2. 语义类切分（一句话理解）

| 方法 | 一句话 | 何时用 |
|------|--------|--------|
| **Semantic Chunking** | 把句子先向量化，按“语义突变点”切 | 主题变化明显的长文 |
| **Proposition Chunking** | 用 LLM 把文本拆成“最小事实陈述”（每条只含一个断言） | 高精度细粒度问答 |
| **Contextual Chunking** | 为每个 chunk 附一段“它处于什么上下文”的说明 | 碎片缺上下文（第 24 课） |
| **Sentence Window** | 按句存，检索时取前后 N 句 | 长文档（第 24 课） |
| **Parent-Child** | 父块大、子块小，检索子块返回父块 | 细召回+大上下文（第 24 课） |

Semantic Chunking 的核心是找“语义断点”：相邻句子的向量相似度突然变低，说明话题切换了，在此处切。

In [ ]:
# 示意：语义切分的“断点检测”骨架
# 生产实现会用 embedding 算相邻句相似度（这里用桩数据演示逻辑）
import numpy as np

def semantic_breakpoints(sim_scores, threshold=0.6):
    """给定相邻句子的相似度序列，返回应切开的位置（相似度骤降处）"""
    return [i for i, s in enumerate(sim_scores) if s < threshold]

# 假设 5 个相邻句子对的相似度：句1-2 相近，2-3 相近，3-4 突降到 0.2（话题变了），4-5 又相近
sims = [0.95, 0.90, 0.20, 0.92]
cuts = semantic_breakpoints(sims, threshold=0.6)
print('相邻句相似度:', sims)
print('判定的话题切换点(下标):', cuts, '→ 在此处切块')
print('\n真实实现 = 用 Embedding 编码每句 → 算相似度 → 找到这些断点。')

In [ ]:
# 知识点·真调说明：Proposition Chunking —— 让 LLM 把一段长文拆成“一条只含一个断言”的最小事实陈述
# 这类 chunk 由 LLM 生成（而非按长度切），专为高精度、细粒度的问答服务。
_llm_live(
    prompt="""请把下面这段话拆成若干条“最小事实陈述”。要求：每条有且只有一个断言，不推理、不合并、不增删信息，输出为编号列表。
文本：缓存穿透指查询一个不存在的数据，由于缓存里也没有，请求会直接打到数据库。缓解手段有两种：一是布隆过滤器，能提前拦下大多数不存在的 key；二是缓存空值，把空结果也缓存一小段时间。注意布隆过滤器只能降低概率，不能完全杜绝穿透。""",
    system='你是文本切分专家，只输出编号列表，每条一句话，不要输出任何解释。',
    fallback="""1. 缓存穿透指查询一个不存在的数据，缓存中没有时请求会直接打到数据库。
2. 缓解缓存穿透的手段有两种。
3. 手段一是布隆过滤器，能提前拦下大多数不存在的 key。
4. 手段二是缓存空值，把空结果也缓存一小段时间。
5. 布隆过滤器只能降低概率，不能完全杜绝穿透。""",
    temperature=0.1,
)
print('→ LLM 拆出的每一条都是“独立可检索”的原子事实：入库时一条事实对应一个向量，问题命中哪条就精确召回哪条，而不必连带整段。')

In [ ]:
# 知识点·真调说明：Contextual Chunking —— 同一个小碎片，不加上下文 vs 加一句“它在讲什么”，模型回答天差地别
# 用 LLM 为被切下的碎片补一段上下文说明并随块入库，是 24 课 Contextual Retrieval 的思路。
print('① 只有碎片本身（缺上下文）—— 模型看不出这段在讲什么')
_llm_live(
    prompt="""请只依据下面给出的资料回答：这段文字在讲什么系统？
资料：「qps 从 8000 提到 20000」。""",
    system='你是问答助手，只能依据给定资料作答；资料信息不足时明确说“无法从资料判断”，不要自行脑补。',
    fallback="""无法从资料判断：只给出“qps 从 8000 提到 20000”一个数字，既看不出是什么系统，也看不出是哪个环节的指标。""",
    temperature=0.1,
)
print()
print('② 给碎片补上“这段属于哪份文档、在讲什么”的上下文说明再问 —— 能答了')
_llm_live(
    prompt="""请依据下面给出的资料回答：这段在讲什么系统、提升了什么？
资料（含上下文说明）：「本节选自《星云客服机器人 v3 技术方案》，主题是把消息队列从 Redis 换成自研存储并调优；其中写道：qps 从 8000 提到 20000」。""",
    system='你是问答助手，只能依据给定资料作答。',
    fallback="""在讲星云客服机器人 v3 的消息队列升级：把消息队列从 Redis 换成自研存储并调优后，qps 从 8000 提升到 20000。""",
    temperature=0.1,
)
print()
print('→ 碎片缺上下文时，即使被检索命中也无法正确作答；Contextual Chunking 用 LLM 给每个碎片补一句“它在讲什么”，让孤立数字也能被可靠回答。')

## 3. 结构类切分（按文档自身结构走）

| 方法 | 思路 | 适配文档 |
|------|------|---------|
| **Markdown Chunking** | 按 `# / ## / ###` 标题层级切，标题进 chunk | MD 文档 |
| **Code Chunking** | 按函数/类(甚至 AST)切，保持代码可执行语义 | 代码仓库（第 32 课） |
| **Table Chunking** | 表头+行成对保留，别把表拆碎 | Excel/表格 |
| **Hierarchical / Parent-Child** | 两级结构：大块(父)与小块(子)并存 | 长章节 |
| **Document Structure** | 结合第 5 课解析出的 标题/章节/页码 切 | PDF 手册 |

共同点：**利用文档自带的层级做切分锚点**，而不是按字数硬切——chunk 与原文结构对齐后，来源标注也更自然。

## 小结

- **语义类**切分：让一块讲一件事（Semantic/Proposition/Contextual…）；
- **结构类**切分：顺着标题/章节/表格/代码结构切；
- 两者可组合，且都要配合 **metadata**（下一课）才能真正好用。